In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create project folders
base_path = '/content/drive/MyDrive/llm_calibration'
os.makedirs(f'{base_path}/results', exist_ok=True)

print("✅ Google Drive mounted successfully!")
print("📁 Folder structure created at:", base_path)
print("\nFolders created:")
for folder in os.listdir(base_path):
    print(f"  └── {folder}")

Mounted at /content/drive
✅ Google Drive mounted successfully!
📁 Folder structure created at: /content/drive/MyDrive/llm_calibration

Folders created:
  └── results


In [2]:
!pip install transformers datasets accelerate -q

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import requests
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Check GPU
print("🖥️ Device Info:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\n🐍 Torch version: {torch.__version__}")

# Fixed seed - same every time
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n🌱 Random seed fixed at: {SEED}")

# Base path
base_path = '/content/drive/MyDrive/llm_calibration'
print(f"📁 Base path: {base_path}")

🖥️ Device Info:
  CUDA available: True
  GPU: Tesla T4
  Memory: 15.6 GB

🐍 Torch version: 2.10.0+cu128

🌱 Random seed fixed at: 42
📁 Base path: /content/drive/MyDrive/llm_calibration


In [3]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
base_path = '/content/drive/MyDrive/llm_calibration'
pile_save_path = f'{base_path}/pile_50k.csv'

# Delete old file
if os.path.exists(pile_save_path):
    os.remove(pile_save_path)
    print("🗑️ Deleted old PILE file")

# We will collect from 3 different parts:
# Part 1: first 17k (skip 0)
# Part 2: middle 17k (skip 200k)
# Part 3: later 16k (skip 500k)
parts = [
    {'skip': 0,      'target': 17000, 'label': 'Start'},
    {'skip': 200000, 'target': 17000, 'label': 'Middle'},
    {'skip': 500000, 'target': 16000, 'label': 'End'},
]

all_samples = []

for part in parts:
    print(f"\n📥 Collecting from {part['label']} (skip={part['skip']})...")

    dataset = load_dataset(
        "monology/pile-uncopyrighted",
        split="train",
        streaming=True
    )

    samples = []
    skipped = 0
    processed = 0

    for example in dataset:
        # Skip to desired position
        if skipped < part['skip']:
            skipped += 1
            continue

        text = example.get('text', '').strip()
        if len(text) > 100:
            samples.append({
                'text': text,
                'source': example.get('meta', {}).get('pile_set_name', 'unknown')
            })
            processed += 1

        if processed >= part['target']:
            break

        if processed % 5000 == 0 and processed > 0:
            print(f"   Collected {processed}/{part['target']}...")

    print(f"   ✅ Collected {len(samples)} samples from {part['label']}")
    all_samples.extend(samples)

# Shuffle
print(f"\n🔀 Shuffling {len(all_samples)} total samples...")
import random
random.seed(SEED)
random.shuffle(all_samples)

pile_df = pd.DataFrame(all_samples[:50000])
pile_df.to_csv(pile_save_path, index=False)

print(f"\n✅ PILE dataset rebuilt!")
print(f"   Total samples: {len(pile_df)}")
print(f"   Saved to: {pile_save_path}")
print(f"\n📊 Source distribution:")
print(pile_df['source'].value_counts().head(10))
print(f"\nSample previews:")
for i in range(3):
    print(f"\n  Sample {i+1} [{pile_df['source'].iloc[i]}]:")
    print(f"    {pile_df['text'].iloc[i][:150]}")


📥 Collecting from Start (skip=0)...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/17000...
   Collected 10000/17000...
   Collected 15000/17000...
   ✅ Collected 17000 samples from Start

📥 Collecting from Middle (skip=200000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/17000...
   Collected 10000/17000...
   Collected 15000/17000...
   ✅ Collected 17000 samples from Middle

📥 Collecting from End (skip=500000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

   Collected 5000/16000...
   Collected 10000/16000...
   Collected 15000/16000...
   ✅ Collected 16000 samples from End

🔀 Shuffling 50000 total samples...

✅ PILE dataset rebuilt!
   Total samples: 50000
   Saved to: /content/drive/MyDrive/llm_calibration/pile_50k.csv

📊 Source distribution:
source
Pile-CC              14827
StackExchange         8417
PubMed Abstracts      8263
Github                4925
Wikipedia (en)        4866
USPTO Backgrounds     3198
PubMed Central        1550
FreeLaw               1390
ArXiv                  679
DM Mathematics         580
Name: count, dtype: int64

Sample previews:

  Sample 1 [Pile-CC]:
    Like most websites we uses cookies. In order to deliver a personalised, responsive service and to improve the site, we remember and store information 

  Sample 2 [Wikipedia (en)]:
    Hector Beers

Hector George Beers was an English cricketer active from 1914 to 1921 who played for Northamptonshire (Northants). He was born in Potter

  Sample 3 [PubMed A

In [4]:
from datasets import load_dataset
import pandas as pd

print("📥 Loading MMLU dataset...")

# MMLU has 57 subjects, we load all of them
mmlu = load_dataset("cais/mmlu", "all", split="test")

print(f"\n✅ MMLU loaded!")
print(f"   Total samples: {len(mmlu)}")
print(f"   Features: {mmlu.features}")

# Convert to dataframe for easy handling
mmlu_df = pd.DataFrame(mmlu)
print(f"\n📊 Subject distribution (top 10):")
print(mmlu_df['subject'].value_counts().head(10))
print(f"\n   Total unique subjects: {mmlu_df['subject'].nunique()}")

# Preview
print(f"\n📝 Sample preview:")
sample = mmlu_df.iloc[0]
print(f"   Question: {sample['question']}")
print(f"   Choices: {sample['choices']}")
print(f"   Answer:  {sample['answer']} ({sample['choices'][sample['answer']]})")
print(f"   Subject: {sample['subject']}")

# Also load validation set for 5-shot examples
mmlu_val = load_dataset("cais/mmlu", "all", split="validation")
mmlu_val_df = pd.DataFrame(mmlu_val)
print(f"\n✅ MMLU validation set (for 5-shot): {len(mmlu_val_df)} samples")

📥 Loading MMLU dataset...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]


✅ MMLU loaded!
   Total samples: 14042
   Features: {'question': Value('string'), 'subject': Value('string'), 'choices': List(Value('string')), 'answer': ClassLabel(names=['A', 'B', 'C', 'D'])}

📊 Subject distribution (top 10):
subject
professional_law              1534
moral_scenarios                895
miscellaneous                  783
professional_psychology        612
high_school_psychology         545
high_school_macroeconomics     390
elementary_mathematics         378
moral_disputes                 346
prehistory                     324
philosophy                     311
Name: count, dtype: int64

   Total unique subjects: 57

📝 Sample preview:
   Question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
   Choices: ['0', '4', '2', '6']
   Answer:  1 (4)
   Subject: abstract_algebra

✅ MMLU validation set (for 5-shot): 1531 samples


In [5]:
import os
import json
import pandas as pd
import numpy as np
import requests

SEED = 42
np.random.seed(SEED)
base_path = '/content/drive/MyDrive/llm_calibration'
trex_save_path = f'{base_path}/trex_50k.csv'

# Force delete old file
if os.path.exists(trex_save_path):
    os.remove(trex_save_path)
    print("🗑️ Deleted old empty file")

print("📥 Downloading T-REx (KILT) dataset...")
url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
response = requests.get(url, stream=True)

samples = []
target = 50000
raw_lines = 0

for line in response.iter_lines():
    if len(samples) >= target:
        break
    if not line:
        continue

    raw_lines += 1
    try:
        item = json.loads(line)

        input_text = item.get('input', '').strip()
        outputs = item.get('output', [])
        if not outputs:
            continue
        answer = outputs[0].get('answer', '').strip()
        if not answer:
            continue

        if '[SEP]' not in input_text:
            continue

        parts = input_text.split('[SEP]')
        subject = parts[0].strip()
        relation = parts[1].strip() if len(parts) > 1 else ''

        # Natural text prompt
        text = f"{subject}'s {relation} is"
        entity = answer

        samples.append({
            'text': text,
            'entity': entity,
            'subject': subject,
            'relation': relation
        })

    except Exception:
        continue

    if raw_lines % 100000 == 0:
        print(f"   Processed {raw_lines} lines, collected {len(samples)} valid...")

response.close()

trex_df = pd.DataFrame(samples[:target])
trex_df.to_csv(trex_save_path, index=False)

print(f"\n✅ T-REx dataset saved!")
print(f"   Raw lines processed: {raw_lines}")
print(f"   Valid samples: {len(trex_df)}")
print(f"   Saved to: {trex_save_path}")

print(f"\n📊 Sample previews:")
for i in range(3):
    print(f"\n  Sample {i+1}:")
    print(f"    Text:     {trex_df['text'].iloc[i]}")
    print(f"    Entity:   {trex_df['entity'].iloc[i]}")
    print(f"    Subject:  {trex_df['subject'].iloc[i]}")
    print(f"    Relation: {trex_df['relation'].iloc[i]}")

📥 Downloading T-REx (KILT) dataset...

✅ T-REx dataset saved!
   Raw lines processed: 50000
   Valid samples: 50000
   Saved to: /content/drive/MyDrive/llm_calibration/trex_50k.csv

📊 Sample previews:

  Sample 1:
    Text:     Serge Blisko's occupation is
    Entity:   politician
    Subject:  Serge Blisko
    Relation: occupation

  Sample 2:
    Text:     Little Hohonu River's located in the administrative territorial entity is
    Entity:   West Coast Region
    Subject:  Little Hohonu River
    Relation: located in the administrative territorial entity

  Sample 3:
    Text:     Medal of Military Valour's country is
    Entity:   Canada
    Subject:  Medal of Military Valour
    Relation: country


In [6]:
import pandas as pd

base_path = '/content/drive/MyDrive/llm_calibration'

mmlu_save_path = f'{base_path}/mmlu_test.csv'
mmlu_val_save_path = f'{base_path}/mmlu_val.csv'

# Save test set
mmlu_df.to_csv(mmlu_save_path, index=False)
print(f"✅ MMLU test set saved: {len(mmlu_df)} samples")

# Save validation set (for 5-shot)
mmlu_val_df.to_csv(mmlu_val_save_path, index=False)
print(f"✅ MMLU validation set saved: {len(mmlu_val_df)} samples")

# Verify
print(f"\n📁 Files on Drive:")
import os
for f in os.listdir(base_path):
    size = os.path.getsize(f'{base_path}/{f}') / 1e6
    print(f"   {f} — {size:.1f} MB")

✅ MMLU test set saved: 14042 samples
✅ MMLU validation set saved: 1531 samples

📁 Files on Drive:
   results — 0.0 MB
   pile_50k.csv — 277.7 MB
   trex_50k.csv — 4.3 MB
   mmlu_test.csv — 6.8 MB
   mmlu_val.csv — 0.7 MB


In [7]:
import torch
import numpy as np
import gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# ECE CALCULATION
# ============================================================
def compute_ece(confidences, accuracies, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total_samples = len(confidences)

    for i in range(n_bins):
        low = bin_boundaries[i]
        high = bin_boundaries[i + 1]
        in_bin = np.where(
            (confidences > low) & (confidences <= high)
        )[0]
        if len(in_bin) == 0:
            continue
        bin_acc = np.mean(accuracies[in_bin])
        bin_conf = np.mean(confidences[in_bin])
        bin_weight = len(in_bin) / total_samples
        ece += bin_weight * abs(bin_acc - bin_conf)

    return ece

# ============================================================
# TASK 1: CLM BATCHED
# ============================================================
def evaluate_clm_batched(model, tokenizer, pile_df,
                          n_samples=30000, batch_size=16,
                          max_length=256):
    model.eval()
    all_confidences = []
    all_accuracies = []

    samples = pile_df.sample(n=n_samples, random_state=SEED)
    texts = samples['text'].tolist()

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="CLM"):
            batch_texts = texts[i:i+batch_size]

            for text in batch_texts:
                tokens = tokenizer.encode(
                    text, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(device)

                if tokens.shape[1] < 3:
                    continue

                pos = np.random.randint(1, min(tokens.shape[1], max_length))
                context = tokens[:, :pos]
                true_token = tokens[0, pos].item() if pos < tokens.shape[1] else None
                if true_token is None:
                    continue

                outputs = model(context)
                logits = outputs.logits[0, -1, :]
                probs = torch.softmax(logits, dim=-1)

                pred_token = torch.argmax(probs).item()
                confidence = probs[pred_token].item()

                all_confidences.append(confidence)
                all_accuracies.append(int(pred_token == true_token))

    conf = np.array(all_confidences)
    acc = np.array(all_accuracies)
    ece = compute_ece(conf, acc)
    mean_acc = np.mean(acc)

    return ece, mean_acc, len(conf)

# ============================================================
# TASK 2: FACTS BATCHED
# ============================================================
def evaluate_facts_batched(model, tokenizer, trex_df,
                            n_samples=30000, batch_size=32,
                            max_length=128):
    model.eval()
    all_confidences = []
    all_accuracies = []

    samples = trex_df.sample(n=n_samples, random_state=SEED)
    texts = samples['text'].tolist()
    entities = samples['entity'].tolist()

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Facts"):
            batch_texts = texts[i:i+batch_size]
            batch_entities = entities[i:i+batch_size]

            for text, entity in zip(batch_texts, batch_entities):
                entity = str(entity).strip()
                text = str(text).strip()

                if not text:
                    continue

                entity_tokens_space = tokenizer.encode(
                    ' ' + entity, add_special_tokens=False
                )
                entity_tokens_no_space = tokenizer.encode(
                    entity, add_special_tokens=False
                )

                if not entity_tokens_space:
                    continue

                true_first_token = entity_tokens_space[0]

                input_ids = tokenizer.encode(
                    text, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(device)

                if input_ids.shape[1] == 0:
                    continue

                outputs = model(input_ids)
                logits = outputs.logits[0, -1, :]
                probs = torch.softmax(logits, dim=-1)

                pred_token = torch.argmax(probs).item()
                confidence = probs[pred_token].item()

                is_correct = int(
                    pred_token == true_first_token or
                    pred_token == entity_tokens_no_space[0]
                )

                all_confidences.append(confidence)
                all_accuracies.append(is_correct)

    if len(all_confidences) == 0:
        return 0.0, 0.0, 0

    conf = np.array(all_confidences)
    acc_arr = np.array(all_accuracies)
    ece = compute_ece(conf, acc_arr)
    mean_acc = np.mean(acc_arr)

    return ece, mean_acc, len(conf)

# ============================================================
# TASK 3: MMLU BATCHED
# ============================================================
def evaluate_mmlu_batched(model, tokenizer, mmlu_df, mmlu_val_df,
                           n_samples=500, n_shot=5, max_length=512):
    model.eval()
    all_confidences = []
    all_accuracies = []

    # Build 5-shot prefix
    few_shot_examples = mmlu_val_df.sample(n=n_shot, random_state=SEED)
    few_shot_text = ""
    for _, ex in few_shot_examples.iterrows():
        choices = ex['choices']
        correct_idx = ex['answer']
        few_shot_text += f"Question: {ex['question']}\n"
        few_shot_text += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
        few_shot_text += f"Answer: {chr(65 + correct_idx)}\n\n"

    samples = mmlu_df.sample(n=n_samples, random_state=SEED)

    # Option token IDs
    option_token_ids = [
        tokenizer.encode(' A', add_special_tokens=False)[-1],
        tokenizer.encode(' B', add_special_tokens=False)[-1],
        tokenizer.encode(' C', add_special_tokens=False)[-1],
        tokenizer.encode(' D', add_special_tokens=False)[-1],
    ]

    # Build prompts
    prompts = []
    correct_indices = []
    for idx, row in samples.iterrows():
        choices = row['choices']
        correct_idx = row['answer']
        prompt = few_shot_text
        prompt += f"Question: {row['question']}\n"
        prompt += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
        prompt += "Answer:"
        prompts.append(prompt)
        correct_indices.append(correct_idx)

    batch_size = 8

    with torch.no_grad():
        for i in tqdm(range(0, len(prompts), batch_size), desc="MMLU"):
            batch_prompts = prompts[i:i+batch_size]
            batch_correct = correct_indices[i:i+batch_size]

            for prompt, correct_idx in zip(batch_prompts, batch_correct):
                input_ids = tokenizer.encode(
                    prompt, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(device)

                if input_ids.shape[1] == 0:
                    continue

                outputs = model(input_ids)
                logits = outputs.logits[0, -1, :]

                option_logits = logits[option_token_ids]
                option_probs = torch.softmax(option_logits, dim=-1)

                pred_idx = torch.argmax(option_probs).item()
                confidence = option_probs[pred_idx].item()
                is_correct = int(pred_idx == correct_idx)

                all_confidences.append(confidence)
                all_accuracies.append(is_correct)

    conf = np.array(all_confidences)
    acc_arr = np.array(all_accuracies)
    ece = compute_ece(conf, acc_arr)
    mean_acc = np.mean(acc_arr)

    return ece, mean_acc, len(conf)

print("✅ All functions defined!")
print("  → compute_ece")
print("  → evaluate_clm_batched")
print("  → evaluate_facts_batched")
print("  → evaluate_mmlu_batched")

✅ All functions defined!
  → compute_ece
  → evaluate_clm_batched
  → evaluate_facts_batched
  → evaluate_mmlu_batched


MODEL 1B

In [8]:
import torch
import numpy as np
import gc
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ============================================================
# CORRECTED MMLU FUNCTION — FULL SEQUENCE SCORING
# ============================================================

def evaluate_mmlu_full(model, tokenizer, mmlu_df, mmlu_val_df, n_samples=None, n_shot=5, max_length=2048, batch_size=8):
    """
    Fixed MMLU with 5-shot in-context examples
    """
    model.eval()
    all_confidences = []
    all_accuracies = []

    # 1. FIX: Set truncation side to 'left' so the actual question isn't cut off
    tokenizer.truncation_side = 'left'

    # Prepare few-shot examples (same for all samples)
    few_shot_examples = mmlu_val_df.sample(n=n_shot, random_state=42)
    few_shot_text = ""
    for _, ex in few_shot_examples.iterrows():
        choices = ex['choices']
        correct_idx = ex['answer']

        few_shot_text += f"Question: {ex['question']}\n"
        few_shot_text += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
        few_shot_text += f"Answer: {chr(65 + correct_idx)}\n\n"

    if n_samples is not None:
        samples = mmlu_df.sample(n=n_samples, random_state=42)
    else:
        samples = mmlu_df

    option_token_ids = [
        tokenizer.encode(' A', add_special_tokens=False)[-1],
        tokenizer.encode(' B', add_special_tokens=False)[-1],
        tokenizer.encode(' C', add_special_tokens=False)[-1],
        tokenizer.encode(' D', add_special_tokens=False)[-1],
    ]

    total = len(samples)

    with torch.no_grad():
        for i in tqdm(range(0, total, batch_size), desc="MMLU Full"):
            batch_samples = samples.iloc[i:i+batch_size]

            for idx, row in batch_samples.iterrows():
                choices = row['choices']
                correct_idx = row['answer']

                prompt = few_shot_text
                prompt += f"Question: {row['question']}\n"
                prompt += f"A. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\n"
                prompt += "Answer:"

                # 2. FIX: Use max_length of 2048 (Pythia's context window)
                input_ids = tokenizer.encode(
                    prompt, return_tensors='pt',
                    max_length=max_length, truncation=True
                ).to(model.device)

                if input_ids.shape[1] == 0:
                    continue

                outputs = model(input_ids)
                logits = outputs.logits[0, -1, :]

                option_logits = logits[option_token_ids]
                option_probs = torch.softmax(option_logits, dim=-1)

                pred_idx = torch.argmax(option_probs).item()
                confidence = option_probs[pred_idx].item()

                is_correct = int(pred_idx == correct_idx)

                all_confidences.append(confidence)
                all_accuracies.append(is_correct)

    conf = np.array(all_confidences)
    acc_arr = np.array(all_accuracies)

    # Ensure compute_ece is defined elsewhere in your notebook
    ece = compute_ece(conf, acc_arr)
    mean_acc = np.mean(acc_arr)

    return ece, mean_acc, len(conf)

# ============================================================
# EVALUATE PYTHIA-1B — ONE MODEL
# ============================================================

model_name = "EleutherAI/pythia-1b"
revision = "step143000"  # Final checkpoint
base_path = '/content/drive/MyDrive/llm_calibration' # Update if your base path changed

print(f"\n{'='*60}")
print(f"🤖 Running PYTHIA-1B")
print(f"{'='*60}")

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    revision=revision,
    torch_dtype=torch.float32,
    device_map='auto'
)
model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Loaded! Params: {n_params:.0f}M | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Task 1: CLM
print(f"\n📊 Task 1: CLM (n=30000)...")
ece_clm, acc_clm, n_clm = evaluate_clm_batched(
    model, tokenizer, pile_df, n_samples=30000, batch_size=8
)
print(f"   ECE: {ece_clm:.4f} | ACC: {acc_clm:.4f} | N: {n_clm}")

# Task 2: Facts
print(f"\n📊 Task 2: Facts Generation (n=30000)...")
# LOWERED BATCH SIZE TO 8 to prevent OOM errors for the 1B model
ece_facts, acc_facts, n_facts = evaluate_facts_batched(
    model, tokenizer, trex_df, n_samples=30000, batch_size=8
)
print(f"   ECE: {ece_facts:.4f} | ACC: {acc_facts:.4f} | N: {n_facts}")

# Task 3: MMLU
print(f"\n📊 Task 3: MMLU (Full Dataset)...")
# FIXED: Calling evaluate_mmlu_full and using n_samples=None
ece_mmlu, acc_mmlu, n_mmlu = evaluate_mmlu_full(
    model, tokenizer, mmlu_df, mmlu_val_df, n_samples=None
)
print(f"   ECE: {ece_mmlu:.4f} | ACC: {acc_mmlu:.4f} | N: {n_mmlu}")

# Save result
result_1b = {
    'model': model_name,
    'revision': revision,
    'n_params_M': round(n_params, 1),
    'ece_clm': round(ece_clm, 4),
    'acc_clm': round(acc_clm, 4),
    'ece_facts': round(ece_facts, 4),
    'acc_facts': round(acc_facts, 4),
    'ece_mmlu': round(ece_mmlu, 4),
    'acc_mmlu': round(acc_mmlu, 4)
}

# Load existing results or create new
results_path = f'{base_path}/phase1_results.csv'
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
else:
    results_df = pd.DataFrame()

# Append new result
results_df = pd.concat([results_df, pd.DataFrame([result_1b])], ignore_index=True)
results_df.to_csv(results_path, index=False)

print(f"\n💾 Results saved to {results_path}!")
print(f"\n📋 Updated Results:")
print(results_df[['model','n_params_M',
                   'ece_clm','acc_clm',
                   'ece_facts','acc_facts',
                   'ece_mmlu','acc_mmlu']].to_string(index=False))

# Cleanup
del model
torch.cuda.empty_cache()
gc.collect()
print(f"\n🧹 GPU cleared: {torch.cuda.memory_allocated()/1e9:.2f} GB")


🤖 Running PYTHIA-1B


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

   ECE: 0.0041 | ACC: 0.5400 | N: 30000

📊 Task 2: Facts Generation (n=30000)...


Facts: 100%|██████████| 3750/3750 [16:07<00:00,  3.88it/s]


   ECE: 0.0852 | ACC: 0.0164 | N: 30000

📊 Task 3: MMLU (Full Dataset)...


MMLU Full: 100%|██████████| 1756/1756 [2:39:52<00:00,  5.46s/it]


   ECE: 0.0825 | ACC: 0.2476 | N: 14042

💾 Results saved to /content/drive/MyDrive/llm_calibration/phase1_results.csv!

📋 Updated Results:
               model  n_params_M  ece_clm  acc_clm  ece_facts  acc_facts  ece_mmlu  acc_mmlu
EleutherAI/pythia-1b      1011.8   0.0041     0.54     0.0852     0.0164    0.0825    0.2476

🧹 GPU cleared: 0.01 GB
